# 01_monitoreo_y_evidencias
Consultas para el README: conector de Tracking (2 corridas), event log del pipeline, expectativas,
historial del job, costos, masking y lineage. Ejecutar sobre el catálogo del target.

In [0]:
dbutils.widgets.text("catalog", "electrocasa_dev")
CATALOG = dbutils.widgets.get("catalog")

## 1. Tracking: evidencia de ingesta incremental
1ª corrida: un `MERGE` con 15.139 filas. 2ª corrida sin cambios en la fuente: no agrega una versión nueva.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.bronze.tracking_envios")
        .select("version", "timestamp", "operation", "operationMetrics"))
display(spark.sql(f"SELECT COUNT(*) AS filas, COUNT(DISTINCT evento_id) AS eventos, MAX(fecha_modificacion) AS cursor_max FROM {CATALOG}.bronze.tracking_envios"))

## 2. Event log del pipeline: corridas y calidad de datos (expectativas)

In [0]:
el = f"event_log(TABLE({CATALOG}.silver.ventas))"
display(spark.sql(f"""
SELECT timestamp, event_type, message
FROM {el}
WHERE event_type IN ('create_update', 'update_progress')
ORDER BY timestamp DESC LIMIT 50"""))

In [0]:
display(spark.sql(f"""
WITH e AS (
  SELECT timestamp, origin.flow_name AS flujo,
         explode(from_json(details:flow_progress:data_quality:expectations,
                 'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS x
  FROM event_log(TABLE({CATALOG}.silver.ventas))
  WHERE event_type = 'flow_progress' AND details:flow_progress:data_quality IS NOT NULL)
SELECT x.dataset, x.name AS expectativa, SUM(x.passed_records) AS pasaron, SUM(x.failed_records) AS fallaron
FROM e GROUP BY ALL ORDER BY x.dataset, expectativa"""))

## 3. Historial de corridas del job (system.lakeflow)
Si la tabla no está disponible, el error se documenta en el README y se usa el historial de corridas de la UI.

In [0]:
%sql
SELECT j.name AS job, t.run_id, t.period_start_time, t.period_end_time, t.result_state,
       timestampdiff(SECOND, t.period_start_time, t.period_end_time) AS segundos
FROM system.lakeflow.job_run_timeline t
JOIN system.lakeflow.jobs j USING (workspace_id, job_id)
WHERE j.name ILIKE '%electrocasa%'
ORDER BY t.period_start_time DESC
LIMIT 50

## 4. (Reto) Costos medidos por recurso — system.billing.usage

In [0]:
%sql
SELECT COALESCE(usage_metadata.job_name, usage_metadata.dlt_pipeline_id, usage_metadata.job_id) AS recurso,
       billing_origin_product, sku_name,
       ROUND(SUM(usage_quantity), 3) AS dbus
FROM system.billing.usage
WHERE usage_date >= DATE'2026-11-18'
  AND (usage_metadata.job_name ILIKE '%electrocasa%'
       OR usage_metadata.dlt_pipeline_id IN (SELECT pipeline_id FROM system.lakeflow.pipelines WHERE name ILIKE '%electrocasa%'))
GROUP BY ALL ORDER BY dbus DESC

## 5. Masking: valores reales para ingeniería en una tabla derivada (gold.empleados_actuales)

In [0]:
display(spark.sql("SELECT current_user() AS usuario, is_account_group_member('electrocasa_ingenieria') AS es_ingenieria"))
display(spark.sql(f"SELECT id_empleado, nombre, dni, salario, sucursal_id FROM {CATALOG}.gold.empleados_actuales LIMIT 10"))

## 6. Respuestas a las 6 preguntas de negocio

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.ventas_netas_sucursal_mes ORDER BY mes, sucursal_id"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.top10_productos_unidades ORDER BY ranking"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.top10_productos_tasa_devolucion ORDER BY ranking"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.dotacion_sucursal_mes ORDER BY fecha_corte, sucursal_id"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.ventas_vendedor_sucursal_mes ORDER BY mes, vendedor_id"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.resenas_negativas_categoria_mes ORDER BY categoria, mes"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.despachos_courier ORDER BY courier"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.gold.resenas_negativas_origen ORDER BY origen, app_version"))

## 7. Lineage (evidencia para Auditoría)

In [0]:
%sql
SELECT source_table_full_name, target_table_full_name, entity_type, MAX(event_time) AS ultimo
FROM system.access.table_lineage
WHERE target_table_catalog LIKE 'electrocasa%'
GROUP BY ALL ORDER BY target_table_full_name